In [ ]:
from pathlib import Path

cwd = Path.cwd().resolve()
ROOT = next(
    (p for p in (cwd, *cwd.parents) if (p / "notebooks").is_dir()),
    None,
)
if ROOT is None:
    raise FileNotFoundError("Nie znaleziono katalogu notebooks")

NOTEBOOKS_DIR = ROOT / "notebooks"
notebooks = []
for path in NOTEBOOKS_DIR.iterdir():
    if path.is_file() and path.suffix == ".ipynb":
        try:
            number = int(path.name[:2])
        except ValueError:
            continue
        notebooks.append((number, path))

notebooks.sort(key=lambda item: (item[0], item[1].name))

print("Lista notebooków do wykonania:")
for number, path in notebooks:
    print(f"{number:02d}: {path.name}")
if not notebooks:
    print("Brak notebooków z numerem w pierwszych dwóch znakach nazwy.")


In [ ]:
import nbformat
from nbclient import NotebookClient

if "notebooks" not in globals():
    raise RuntimeError("Najpierw uruchom pierwszą komórkę z listowaniem.")

for _, path in notebooks:
    print(f"\nURUCHAMIAM: {path.name}", flush=True)
    notebook = None
    try:
        notebook = nbformat.read(path, as_version=4)
        client = NotebookClient(
            notebook,
            timeout=None,
            kernel_name="python3",
            allow_errors=False,
            resources={"metadata": {"path": str(NOTEBOOKS_DIR)}},
        )
        client.execute()
        nbformat.write(notebook, path)
    except Exception as exc:
        print(f"\nBŁĄD W NOTEBOOKU: {path.name}\n{exc}", flush=True)
        if notebook is not None:
            try:
                nbformat.write(notebook, path)
            except Exception as save_error:
                print(f"Nie udało się zapisać wyników: {save_error}", flush=True)
        print("Zatrzymano. Kolejne notebooki nie zostaną uruchomione.", flush=True)
        break
    except KeyboardInterrupt:
        print(f"\nPrzerwano: {path.name}. Kolejne notebooki nie zostaną uruchomione.", flush=True)
        break
    print(f"OK: {path.name}", flush=True)
else:
    if notebooks:
        print("\nWszystkie notebooki wykonane.", flush=True)
    else:
        print("Lista jest pusta — nic do uruchomienia.", flush=True)
